# 03 Modeling & performance reporting
이미 실행된 결과가 포함되어 있다. `RETRAIN=True`일 때만 전체 8개 후보·nested CV를 재실행한다. Hold-out/test로 재튜닝하지 않는다.

Colab: 프로젝트 ZIP을 업로드하고 `!unzip -q /content/ess_battery_project.zip -d /content` 후 `%cd /content/ess-battery-project`, `!pip install -r requirements.txt`를 실행한다.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT.parent
if not (ROOT / 'data/battery_modeling_ready.parquet').exists():
    raise FileNotFoundError('프로젝트 ZIP을 풀고 프로젝트 루트 또는 notebooks 폴더에서 실행하세요.')
sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image
from src.preprocess import load_data
df = load_data(ROOT / 'data/battery_modeling_ready.parquet')


In [ ]:
RETRAIN = False
if RETRAIN:
    from src.train import run
    run(ROOT / 'data/battery_modeling_ready.parquet', ROOT / 'results', n_jobs=2)
    from src.analyze import run as analyze
    analyze(ROOT)

In [ ]:
import json
selection = json.loads((ROOT / 'results/selection.json').read_text())
print(json.dumps(selection, ensure_ascii=False, indent=2))
display(pd.read_csv(ROOT / 'results/cv_comparison.csv'))
display(pd.read_csv(ROOT / 'results/performance_reporting.csv'))
display(pd.read_csv(ROOT / 'results/model_performance.csv'))

In [ ]:
for name in ['cv_comparison','actual_vs_predicted','error_diagnostics','external_comparison']:
    display(Image(filename=str(ROOT / f'results/figures/{name}.png')))
display(pd.read_csv(ROOT / 'results/error_analysis.csv').head(10))
display(pd.read_csv(ROOT / 'results/error_summary.csv'))

In [ ]:
# 본 프로젝트에서 생성한 신뢰할 수 있는 파일만 로드한다.
import joblib
model = joblib.load(ROOT / 'results/final_model.joblib')
new_cells = df.loc[df.split.eq('test1')].copy()
new_cells['prediction'] = model.predict(new_cells)
display(new_cells[['battery_id','cycle_life','prediction','has_label']])

Train CV=선택 절차의 외부 fold 평균(최종 SVR 단독 고정 설정의 CV가 아님). 9.204%는 선택에 사용한 내부 CV이고 13.489%는 선택 절차 nested CV다. 후보 외부 CV를 보고 최종 모델을 사후 변경하지 않았다. 9.1%는 과제 목표이며 동일 실험 조건의 논문 재현 점수와 혼동하지 않는다.